## Hands-On Lab 3 - Lags

In this lab, you will craft your first machine learning forecasting model (i.e., a random forest) using lagged features. You will then evaluate the ML forecasting model and compare it to the best baseline model (i.e., the revised naive forecast model).

### Step 1 - Loading the Training Data

Run the following code to load the Bike Sharing training data and set the index.

In [ ]:
import pandas as pd

# Load training data
bike_sharing_train = pd.read_csv('BikeSharing_Train.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_train = bike_sharing_train.set_index('date_time')
bike_sharing_train = bike_sharing_train.asfreq('h')
bike_sharing_train = bike_sharing_train.sort_index()

### Step 2 - Loading the Validation Data

Run the following code to load the Bike Sharing validation data and set the index.

In [ ]:
# Load validation data
bike_sharing_val = pd.read_csv('BikeSharing_Validation.csv', parse_dates = ['date_time'])

# Set index
bike_sharing_val = bike_sharing_val.set_index('date_time')
bike_sharing_val = bike_sharing_val.asfreq('h')
bike_sharing_val = bike_sharing_val.sort_index()

### Step 3 - Chart Set Up

Run the following code to set up line charts using dark mode.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from skforecast.plot import set_dark_theme

# Set fonts/grid defaults
custom_style = {
    'axes.titlesize': 18,
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'grid.alpha': 0.3,
}

# Make dark mode the default for all plots
set_dark_theme(custom_style = custom_style) 

### Step 4 - Forecast KPI Calculations

Run the following code for the utility function that calculates forecasting model KPIs (i.e., bias and MAE).

In [ ]:
import numpy as np

# Utility function for our model KPIs
def forecast_kpis(forecasts, targets):
    # Bias calculations
    errors = forecasts - targets 
    targets_sum = np.sum(targets)
    bias = np.mean(errors)
    bias_pct = np.sum(errors) / targets_sum 

    # MAE calculations
    mae = np.mean(np.abs(errors))
    mae_pct = np.sum(np.abs(errors)) / targets_sum

    # Display
    print(f'Bias: {bias:0.4f}, {(bias_pct * 100):0.2f}%')
    print(f'MAE: {mae:0.4f}, {(mae_pct * 100):0.2f}%') 

### Step 5 - Baseline Forecasting Model

Run the following code to train a naive forecasting model that uses the number of users for the same hour from the previous day (i.e., the third model from the last lab).

In [ ]:
from skforecast.recursive._forecaster_equivalent_date import ForecasterEquivalentDate

# Forecast horizon
h = 168

# A naive forecasting model using the previous day's users for the same 
# hour for predictions
naive_last_day = ForecasterEquivalentDate(offset = pd.DateOffset(days = 1), 
                                          n_offsets = 1)

# Learn from the data
naive_last_day.fit(y = bike_sharing_train['users'])

# Make predictions
naive_preds_last_day = naive_last_day.predict(steps = h)

# Get KPIs
forecast_kpis(naive_preds_last_day, bike_sharing_val['users'].head(n = h))

### Step 6 - Finding the Lags

As you learned in lecture, you can use a partial autocorrelation (PACF) plot to help you choose an initial set of "optimal" parameters. Enter and run the following code to display the PACF plot for the Bike Sharing training dataset.

In [ ]:
# Enter your lab code here

### Step 7 - Training the ML Forecasting Model

As you learned in lecture, lagged features are the foundation for any machine learning forecasting model. Not surprisingly, selecting the optimal lags is crucial for developing effective ML forecasting models. Enter and run the following code that uses "optimal" lags found using the PACF plot in the last step.

In [ ]:
# Enter your lab code here

### Step 8 - Visualize the RF Lagged Model

Run the following code to visualize the random forest (RF) lagged forecasts versus the validation dataset observations.

In [ ]:
# Create a plot 11 inches wide and 5 inches high
fig, ax = plt.subplots(figsize = (11, 5))

# Validation targets
bike_sharing_val['users'].head(n = h).plot(ax = ax, linewidth = 2)

# Forecasts
rf_pacf_lags_preds.plot(ax = ax, linewidth = 2, linestyle = '--', 
                        label = 'RF Lagged Forecast')

# Display plot
ax.set_title('Random Forest Lagged Forecasts for Validation Dataset')
ax.legend()
fig.tight_layout()
plt.show()

### Step 9 - "Brute Force" RF Lagged Model

Run the following code to evaluate a model that uses all lags ranging from 1 to 25 as a comparison.

In [ ]:
# Forecast using a Random Forest using all lags from 1 to 25
rf_all_lags = ForecasterRecursive(regressor = RandomForestRegressor(random_state = 12345), lags = 25)

# Learn from the lagged dataset
rf_all_lags.fit(y = bike_sharing_train['users'])

# Make predictions to compare against validation set
rf_all_lags_preds = rf_all_lags.predict(steps = h)

# What is the accuracy on the validation set?
forecast_kpis(rf_all_lags_preds, bike_sharing_val['users'])